<h1 style="text-align: center">Getting Started into Computer Vision</h1>


Visi komputer / Computer vision adalah metode yang memungkinkan komputer dan sistem bertindak berdasarkan informasi yang diperoleh dari gambar dan video. Computer Vision menggabungkan komputasi edge, komputasi cloud, perangkat lunak, dan model deep learning AI untuk mengenali, mengklasifikasikan, menganalisis, dan mengambil tindakan berdasarkan data visual yang dikumpulkan


### Preparing Dependencies
Install opencv Python <br> Jalankan perintah berikut di Command Line : <br></br>
`pip install numpy` <br>
`pip install opencv-python` <br>
`pip install matplotlib`<br>
`pip install ultralytics`<br>
`sudo apt install ros-noetic-cv-bridge ros-noetic-image-transport`

check installation
script python
```python
import cv2
print(cv2.__version__)
```
or
from shell
```bash
pip list | grep cv
sudo apt search ros-noetic | grep -E "cv-bridge|image-transport"
```


Pada Sesi kali ini menggunakan library OpenCV (Open Source Computer Vision Library) yang mana sebuah library open source yang dikembangkan oleh intel yang fokus untuk menyederhanakan programing terkait citra digital.

OpenCV sudah mempunyai banyak fitur, antara lain : pengenalan wajah, pelacakan wajah, deteksi wajah, Kalman filtering, dan berbagai jenis metode AI (Artificial Intellegence).[1]


In [19]:
!pip list | grep cv


opencv-python               5.0.0.93
opencv-python-headless      4.10.0.84


OpenCV udah punya toolbox lengkap:
- Filtering, thresholding, dan transformasi (blur, edge, morphology)
- Face/object detection (Haar, DNN)
- Optical flow, feature matching, contour detection
- Camera calibration, stereo vision, dll

OpenCV dapat di integrasikan dengan berbagai hal contoh:
- Numpy
- ROS
- TensorFlow / Torch
- OpenGL
- etc


### Read Image


In [21]:
import cv2

img = cv2.imread('samples/images/sample_image.jpeg')
cv2.imshow('image',img)
cv2.waitKey(0)
cv2.destroyAllWindows()


pada code tersebut dilakukan untuk membaca suatu gambar. Dimana fungsi `cv2.imread(img)` untuk read gambar yang mana img merupakan path menuju ke gambar.  Gambar haruslah berada dalam satu directory folder atau directory yang lengkap. Kalau gambarnya nggak bisa dibaca entah karena filenya hilang, izin aksesnya nggak ada, atau formatnya aneh/tidak didukung maka hasilnya bakal kosong (cuma matriks kosong doang).

untuk menampilkan menggunakan fungsi `cv2.imshow('nama window', variable_img)` berfungsi untuk menampilkan gambar pada window baru.

penggunaan `cv2.waitKey(0)` di gunakan untuk mengambil / menunggu tombol apapun yang ter tekan akan mengeksekusi

untuk menutup window dari `imshow()` disarankan menggunakan `cv2.destroyAllWindows()`


### Read Video


- via file


In [6]:
import cv2

cap = cv2.VideoCapture("samples/videos/sample_video.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow("Video", frame)
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break   

cap.release()
cv2.destroyAllWindows()


Objek VideoCapture akan memberi kita frame satu per satu setiap kali kita memanggil .read().

VideoCapture akan menampilkan video hingga durasi video habis (pada contoh modul 10 detik)


- via capture camera


In [23]:
import cv2

cap = cv2.VideoCapture("samples/videos/sample_video.mp4")  # 0 = kamera default

if not cap.isOpened():
    print("Error: Tidak dapat membuka kamera.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Gagal membaca frame dari kamera.")
        break

    cv2.imshow("Webcam Live", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Dihentikan oleh pengguna.")
        break

cap.release()
cv2.destroyAllWindows()


Dihentikan oleh pengguna.


Input dari Video capture sendiri bisa berupa path-file, index camera, path index(/dev/<camera index>),maupun stream IP Contoh:
```python
cv2.VideoCapture(0)
cv2.VideoCapture(1) untuk kamera USB
cv2.VideoCapture("http://192.168.0.10:8080/video")
```

kita juga dapat mengatur resolusi output dari video maupun gambar yang dibaca. Contoh:
```python
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# check parameter
print("Resolusi :", cap.get(cv2.CAP_PROP_FRAME_WIDTH), "x", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("FPS :", cap.get(cv2.CAP_PROP_FPS))
```


### Manipulasi hasil pembacaan gambar
Pada OpenCV, terdapat berbagai fungsi yang dapat digunakan untuk memanipulasi gambar yang diperoleh melalui proses pemrosesan citra

Untuk melakukan resize dengan fungsi
```python
cv.resize()
```
inputnya harus berupa integer bukan floating number. itulah kenapa int() atau fungsi round() dapat di gunakan.


In [ ]:
import cv2

img = cv2.imread("samples/images/sample_image.jpeg")
original_height, original_width, channels = img.shape

scale = 1

new_height = int(original_height * scale)
new_width = int(original_width * scale)

resized = cv2.resize(img, (100, 500))
cv2.imshow("Resized", resized)
cv2.waitKey(0)
cv2.destroyAllWindows()


nah fungsi terdapat fitur ROI (Region of Interest) sama kayak penggunaan zoom dengan data yang hanya inginkan saja


In [33]:
import cv2

img = cv2.imread("samples/images/sample_image.jpeg")

# Ambil ROI (y: 50–250, x: 100–400) → tinggi 200, lebar 300
roi = img[50:250, 100:400]

# Tempel ROI ke posisi lain
img_copy = img.copy()
img_copy[0:200, 0:300] = roi

cv2.imshow("ROI", roi)
cv2.imshow("Pasted ROI", img_copy)
cv2.waitKey(0)
cv2.destroyAllWindows()


OpenCV bisa merotasi gambar karena dia pakai transformasi geometrik, tepatnya Affine Transformation.
Rotasi itu cuma salah satu bentuk affine transform/dia ngubah posisi tiap piksel berdasarkan rumus matematis rotasi.

more explanation for rotation: https://www.geeksforgeeks.org/computer-vision/python-opencv-getrotationmatrix2d-function/


In [29]:
center = (img.shape[1] // 2, img.shape[0] // 2)
angle = 45
scale = 1.0
rotation_matrix = cv2.getRotationMatrix2D(center, angle, scale)
maniped = cv2.warpAffine(img, rotation_matrix, (img.shape[1], img.shape[0]))

cv2.imshow("Manipulated", maniped)
cv2.waitKey(0)
cv2.destroyAllWindows()


jadi output dari getRotationMatrix2D isinya rumus matematis buat muter + nge-scale + mindahin posisi gambar.


Melakukan penggambaran geometri pada openCV dapat di lakukan dengan fungsi-fungsi di bawah ini


In [36]:
cv2.line(img,(0,0),(511,511), (255,0,0), 5)
cv2.circle(img, (250,250), 63, (50,50,0), -1)
cv2.rectangle(img,(384,0),(510,128),(0,255,0),3)
cv2.putText(img, 'OpenCV', (10,200), cv2.FONT_HERSHEY_SIMPLEX, 4, (255,255,255), 5)

cv2.imshow("Edited", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


### Manipulasi warna & bitwise

Ruang warna itu cara menggambarkan dan menyimpan warna dalam gambar digital setiap “ruang” punya struktur berbeda (misalnya RGB, HSV, Grayscale)

secara default OpenCV akan menggunakan BGR bukan RGB


In [37]:
import cv2
import numpy as np

gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv      = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
rgb      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

gray_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

all_img  = np.concat([img, gray_rgb, hsv, rgb], axis=1)
cv2.imshow("Many channels", all_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


@Funfact
kalau di coba
```python
print(img.shape)
```

outputnya = y, x, channel

pewarnaan gray hanya mengeluarkan height dan width saja channelnya tidak ada.
sedangkan channel itu apa?

Channel = lapisan data yang menyimpan intensitas untuk satu komponen warna.
Setiap piksel di gambar punya beberapa channel tergantung ruang warnanya.

Bayangin gambar warna itu kayak tumpukan tiga kertas transparan:
- satu kertas buat warna biru (B),
- satu buat hijau (G),
- satu buat merah (R).

Kalau ditumpuk bareng, jadilah gambar berwarna penuh.
Itu sebabnya gambar warna punya 3 channel.


##### Bitwise
Operasi “bitwise” artinya operasi logika tingkat bit antara dua gambar (atau gambar + mask) yang mempunyai dimensi dan tipe data sama.
Digunakan buat misalnya “masking” objek, ekstraksi bagian penting dari gambar, atau kombinasi dua citra


In [38]:
import cv2

img1 = cv2.imread('samples/images/sample_image1.jpg')
img2 = cv2.imread('samples/images/sample_image2.png')

dest_and = cv2.bitwise_and(img2, img1, mask = None)
dest_or  = cv2.bitwise_or(img2, img1, mask = None)
dest_xor = cv2.bitwise_xor(img2, img1, mask = None)
dest_not = cv2.bitwise_not(img2, mask = None)

img_concat = np.concat([dest_and, dest_or, dest_xor, dest_not], axis=1)

cv2.imshow("Bitwise", img_concat)
cv2.waitKey(0)
cv2.destroyAllWindows()

##### Flood Fill Algorithm
Kayak banjir yang menerjang, flood fill algorithm mulai di satu titik terus nyebar ke titik lain yang punya nilai sama dengan titik awal atau area yang dibatasin oleh suatu nilai. Biasanya dipakai di gambar hitam putih, tetapi gambar RGB juga bisa (distance-based).

In [39]:
import cv2
import numpy as np

# bikin gambar sampel
binary_img = np.zeros((224, 224)).astype(np.uint8)
binary_img[50:200, 50:200] += 255
binary_img[100:150, 100:150] = 0

# flood fill
h, w = binary_img.shape[:2]
binary_copy = binary_img.copy()
center_pt = (0, 0)
fill_value = 255
fill_img = np.zeros((h+2, w+2), np.uint8)
cv2.floodFill(binary_copy, fill_img, center_pt, fill_value)

# inverted flood fill
flood_fill_inv = ~binary_copy

# combined inverted flood fill and original
combined_flood_fill_original = flood_fill_inv | binary_img

cv2.imshow(
    "Original vs Flood Fill vs Inverted Flood Fill vs Combined Original and Inverted", 
    np.concat([binary_img, binary_copy, flood_fill_inv, combined_flood_fill_original], axis=1)
)
cv2.waitKey(0)
cv2.destroyAllWindows()

##### Transformasi Morfologi (Morphological Transformation)

Transformasi morfologi adalah operasi pada citra yang memanipulasi bentuk/struktur objek, biasanya pada citra biner atau grayscale setelah proses thresholding. Tujuannya adalah memperbaiki hasil segmentasi, menghapus noise, atau menutup lubang pada objek.

Operasi dasar dalam morfologi:
- Erosion (erosi): mengecilkan objek, berguna untuk menghilangkan piksel kecil/noise
- Dilation (dilasi): memperbesar objek, berguna untuk menutup celah
- Opening: erosi lalu dilasi, cocok untuk menghapus noise kecil
- Closing: dilasi lalu erosi, cocok untuk menutup lubang/cekungan

In [40]:
import cv2
import numpy as np

img = cv2.imread("samples/images/sample_image.jpeg", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (300, 200))
_, mask = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

kernel = np.ones((5, 5), np.uint8)

eroded = cv2.erode(mask, kernel, iterations=1)
dilated = cv2.dilate(mask, kernel, iterations=1)
opening = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

cv2.imshow("Original vs Erosion vs Dilation vs Opening vs Closing",
           np.concat([mask, eroded, dilated, opening, closing], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

# [1 0 0]
# [1 1 1]
# [0 1 1]

##### Color Thresholding

Thresholding warna biasanya dilakukan pada ruang warna HSV, karena HSV memisahkan informasi warna dari kecerahan. Ini lebih stabil untuk deteksi warna dibanding RGB.

Contoh: mendeteksi objek berwarna merah:

In [41]:
import cv2
import numpy as np

img = cv2.imread("samples/images/sample_image1.jpg")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

lower_red = np.array([0, 50, 45])
upper_red = np.array([15, 255, 255])

mask = cv2.inRange(hsv, lower_red, upper_red)
result = cv2.bitwise_and(img, img, mask=mask)

mask_rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)           # konversi ke bgr karena berbeda jumlah channel
cv2.imshow("Original vs Red Mask vs Detected Red",
           np.concat([img, mask_rgb, result], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

##### Thresholding Grayscale

Thresholding biasa dipakai pada citra grayscale, di mana intensitas piksel diubah menjadi hitam-putih berdasarkan nilai ambang tertentu. <br>
Jenis threshold:
- `THRESH_BINARY`: di atas threshold -> putih, bawah -> hitam
- `THRESH_BINARY_INV`: kebalikan dari BINARY
- `THRESH_TRUNC`: nilai di atas threshold dibatasi ke threshold
- `THRESH_TOZERO`: nilai di bawah threshold dibuat 0

In [44]:
import cv2

img = cv2.imread("samples/images/sample_image2.png", cv2.IMREAD_GRAYSCALE)

ret, th1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
ret, th2 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
ret, th3 = cv2.threshold(img, 50, 255, cv2.THRESH_TRUNC)
ret, th4 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)

cv2.imshow(
    "Original vs BINARY vs BINARY INV vs TRUNC vs TOZERO",
    np.concat([img, th1, th2, th3, th4], axis=1)
)
cv2.waitKey(0)
cv2.destroyAllWindows()

##### Gaussian Blur

Gaussian blur adalah proses menghaluskan citra dengan filter Gaussian untuk mengurangi noise dan detail halus sebelum proses thresholding atau deteksi tepi.

In [45]:
import cv2

img = cv2.imread("samples/images/sample_image.jpeg", cv2.IMREAD_GRAYSCALE)

# Ukuran kernel `(5, 5)` atau `(7, 7)` umum dipakai. Semakin besar kernel, semakin halus hasilnya, tetapi detail bisa berkurang.
blur = cv2.GaussianBlur(img, (5, 5), 0)

cv2.imshow("Original vs Gaussian Blur", np.concat([img, blur], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

### Deteksi Tepi (Edge Detection)

Tepi adalah bagian terluar dari sebuah objek pada gambar. Deteksi tepi biasanya dilakukan dengan mencari perubahan warna intens pada gambar.

##### Sobel

Sobel adalah operator gradien yang digunakan untuk mendeteksi perubahan intensitas pada citra, sehingga berguna untuk menemukan tepi.

In [49]:
import cv2

img = cv2.imread("samples/images/sample_image1.jpg", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (300, 200))
blur = cv2.GaussianBlur(img, (5, 5), 0)

sobel_x = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)

sobel_x = cv2.convertScaleAbs(sobel_x)
sobel_y = cv2.convertScaleAbs(sobel_y)

sobel_combined = cv2.addWeighted(sobel_x, 0.5, sobel_y, 0.5, 0)

cv2.imshow("Original vs Sobel X vs Sobel Y vs Sobel Combined",
           np.concat([img, sobel_x, sobel_y, sobel_combined], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

##### Canny

Canny adalah salah satu metode deteksi tepi paling populer. Ia melakukan beberapa tahapan:
1. Gaussian blur untuk mengurangi noise
2. Sobel untuk mencari gradien
3. Non-maximum suppression
4. Hysteresis thresholding

In [50]:
import cv2

img = cv2.imread("samples/images/sample_image1.jpg", cv2.IMREAD_GRAYSCALE)
blur = cv2.GaussianBlur(img, (5, 5), 0)

low_thresh = 50
high_thresh = 150
edges = cv2.Canny(blur, low_thresh, high_thresh)

cv2.imshow("Original vs Canny Edge", np.concat([img, edges], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

### Mencari Kontur (Contour Detection)

Kontur adalah area yang dibatasi oleh tepi. Semua kontur pasti memiliki tepi, tetapi semua tepi belum tentu berisikan kontur. 

In [53]:
import cv2
import numpy as np

img = cv2.imread("samples/images/sample_image2.png", cv2.IMREAD_GRAYSCALE)

# pakai Canny, Sobel, atau thresholding untuk deteksi tepi dulu
blur = cv2.GaussianBlur(img, (5, 5), 0)
_, edges = cv2.threshold(blur, 127, 255, cv2.THRESH_BINARY)

# cari kontur dulu
contour, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# gambar kontur
contour_img = np.zeros_like(img)
for i in range(len(contour)):
    cv2.drawContours(contour_img, contour, i, 255, 1)

cv2.imshow("Grayscale vs Edge vs Contour",
           np.concat([img, edges, contour_img], axis=1))
cv2.waitKey(0)
cv2.destroyAllWindows()

Deteksi saja tidak cukup. Kita juga perlu informasi tentang kontur yang diekstraksi, seperti jumlah sisi dan ukurannya.

In [54]:
import cv2
import numpy as np

SKALA = 0.02        # antara 0 dan 1
img = cv2.imread("samples/images/sample_image2.png", cv2.IMREAD_GRAYSCALE)

# pakai Canny, Sobel, atau thresholding untuk deteksi tepi dulu
blur = cv2.GaussianBlur(img, (5, 5), 0)
_, edges = cv2.threshold(blur, 127, 255, cv2.THRESH_BINARY)

# cari kontur dulu
contour, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

# gambar kontur
for i in range(len(contour)):
    contour_img = np.zeros_like(img)
    cv2.drawContours(contour_img, contour, i, 255, 1)
    
    # approksimasi
    # pakai epsilon atau toleransi untuk garis lurus
    epsilon = SKALA * cv2.arcLength(contour[i], True)
    approx = cv2.approxPolyDP(contour[i], epsilon, True)
    
    # ekstraksi informasi
    jumlah_sisi = len(approx)
    keliling = cv2.arcLength(contour[i], True)
    luas = cv2.contourArea(contour[i])
    
    # ditampilkan
    print(f"\nKontur ke-{i+1}:")
    print(f"  Jumlah sisi: {jumlah_sisi}")
    print(f"  Keliling: {keliling}")
    print(f"  Luas: {luas}")
    print(f"  Titik perpotongan (x, y): {approx.squeeze(1).tolist()}")
    
    cv2.imshow("Contour", contour_img)
    cv2.waitKey(0)
    
cv2.destroyAllWindows()


Kontur ke-1:
  Jumlah sisi: 6
  Keliling: 100.62741661071777
  Luas: 301.0
  Titik perpotongan (x, y): [[230, 178], [230, 189], [227, 192], [210, 194], [221, 213], [230, 213]]

Kontur ke-2:
  Jumlah sisi: 7
  Keliling: 88.18376553058624
  Luas: 179.5
  Titik perpotongan (x, y): [[195, 167], [200, 172], [211, 176], [206, 181], [214, 183], [226, 181], [218, 173]]

Kontur ke-3:
  Jumlah sisi: 7
  Keliling: 50.041630148887634
  Luas: 43.5
  Titik perpotongan (x, y): [[89, 162], [89, 166], [97, 177], [103, 178], [97, 176], [94, 165], [91, 162]]

Kontur ke-4:
  Jumlah sisi: 8
  Keliling: 71.94112491607666
  Luas: 125.0
  Titik perpotongan (x, y): [[133, 159], [129, 160], [126, 168], [120, 174], [110, 176], [117, 178], [126, 175], [133, 164]]

Kontur ke-5:
  Jumlah sisi: 4
  Keliling: 8.828427076339722
  Luas: 3.0
  Titik perpotongan (x, y): [[108, 152], [109, 153], [111, 153], [112, 152]]

Kontur ke-6:
  Jumlah sisi: 12
  Keliling: 259.76449966430664
  Luas: 1160.0
  Titik perpotongan (x, y

### Pengenalan YOLO
YOLO (You Only Look Once) adalah sebuah model object detection yang dirilis pada tahun 2015 dalam karya ilmiah [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640). Tidak seperti R-CNN family, YOLO langsung melakukan agregasi bounding box dan class pada input gambar, menjadikannya sebagai one-stage detector dan mengurangi waktu inferensi secara signifikan. 


#### Kapan Menggunakan YOLO?

Tidak setiap masalah perlu diselesaikan dengan menggunakan YOLO. YOLO memang merupakan model yang sudah teruji oleh berbagai peneliti mampu menghasilkan prediksi objek yang akurat, tetapi masalah yang spesifik justru dapat diselesaikan dengan metode yang lebih simple dan murah dengan hasil yang tidak kalah jauh.

Bayangkan tiga kotak dengan warna yang berbeda diletakkan bersisihan antara satu sama lain. Sebuah robot ditugaskan untuk mengambil kotak yang memiliki warna tertentu. Apakah YOLO diperlukan dalam kasus ini? Tidak, justru deteksi warna saja sudah cukup untuk melakukannya, dengan asumsi warna invarian terhadap lighting dan bayangan. Pada kasus dengan derajat ketidakpastian yang kecil, metode spesifik yang naif biasanya sudah cukup untuk menyelesaikan masalah-masalah tertentu ketika asumsi yang dipakai sesuai dengan kondisi dari permasalahan tertentu.

Contoh tugas yang bisa diselesaikan dengan YOLO adalah deteksi objek dengan geometri dan lingkungan yang tidak pasti (memiliki berbagai bentuk, lighting sangat bervariasi, dan lain sebagainya). Masalah tersebut menjadikan YOLO sebagai solusi yang cocok untuk digunakan.


#### Langkah-langkah Menggunakan YOLO


##### Persiapan Dataset


<hr></hr>

YOLO memiliki sebuah konvensi tersendiri untuk melakukan training. Framework yang dipakai meminta sebuah struktur dataset seperti berikut.
```bash
dataset/
├── train/
│   ├── images/
│   ├── labels/
├── test/
│   ├── labels/
│   ├── images/
├── val/
│   ├── labels/
│   ├── images/
└── data.yaml
```

Folder `images` berisikan gambar-gambar yang dipakai YOLO untuk melakukan training dan prediksi, sedangkan folder `labels` berisikan anotasi dari tiap gambar dalam file `.txt` (FILENAME GAMBAR DAN ANOTASI HARUS SAMA, KECUALI FORMATNYA). Setiap anotasi memuat nilai:
```
[cls_id, cx, cy, w, h]
```
dengan:
- **cls_id** = ID dari class sebenarnya (dalam bentuk integer)
- **cx**     = titik tengah dari bounding box berdasarkan sumbu x (ternormalisasi dengan rentang [0, 1])
- **cy**     = titik tengah dari bounding box berdasarkan sumbu y (ternormalisasi dengan rentang [0, 1])
- **w**      = lebar dari bounding box (ternormalisasi dengan rentang [0, 1])
- **h**      = tinggi dari bounding box (ternormalisasi dengan rentang [0, 1])

---

File `data.yaml` berisikan "blueprint" untuk training, memberikan framework informasi tentang dataset yang dipakai. Isi standar dari `data.yaml` adalah sebagai berikut:
```yaml
# Path
path: ../datasets/dataset_name  # memberitahu letak folder dataset di dalam sistem (opsional)
train: images/train             # memberitahu letak folder train split
val: images/val                 # memberitahu letak folder validation split
test: images/test               # memberitahu letak folder test split (opsional)

# Class
nc: n                           # jumlah class yang ada di dataset
names:                          # mapping antara ID class ke nama class (mulai dari 0)
    0: class0
    1: class1
    2: class2
    3: class3
    # ...
    n: classN
```

---

Setelah membuat dataset, validasi dataset yang ada dengan melakukan data preprocessing terlebih dahulu. Validasi ini bertujuan untuk mendeteksi ketidaksinambungan antara gambar dan anotasi (gambar tanpa anotasi atau sebaliknya), nilai anotasi yang tidak masuk akal (nilai `cx`, `cy`, `w`, dan/atau `h` yang tidak berada di rentang [0, 1], dan ID class yang tidak ada di dalam `data.yaml`), masalah pada loading data (gambar korup dan lain sebagainya), dan masih banyak lagi. YOLO secara internal telah melakukan ini sebelum training dimulai, tetapi data preprocessing secara manual memberikan kontrol terhadap apa yang seharusnya dilakukan pada sebuah data dibandingkan dengan langkah preprocessing YOLO yang cukup naif. 

Untuk melihat file yang ada pada folder, package `os` bisa digunakan.

---

```python
import os

path      = "path/to/folder"

# hanya filename
filenames = os.listdir(path)

# filename dan directory lengkap
filenames = []
for filename in os.listdir(path):
    file_path = os.path.join(path, filename)
    filenames.append(file_path)                 # menambahkan item ke `filenames`, mirip seperti `push_back()` di C++

# filename dan directory lengkap (one-liner)
filenames = [os.path.join(path, file) for file in os.listdir(path)]

# os.listdir() untuk mendapatkan semua filename dari sebuah folder
# os.path.join() menggabungkan dua atau lebih path menjadi satu
```
---

Kemudian, gambar bisa dimuat dengan fungsi `cv2.imread()` milik OpenCV.

Membaca anotasi di dalam file `.txt` bisa dilakukan dengan menggunakan fungsi `open()`.

--- 
```python
txt_path = "path/to/file.txt"

with open(txt_path, "r") as f:
    data = f.read()

# perlu diperhatikan bahwa open() membaca isi dari filenya dalam bentuk string, maka harus dilakukan splitting

data = data.split("\n") # melakukan splitting dengan delimiter baris baru ('\n') dan mereturn sebuah list berisi string

# data masih berupa string, padahal yang dibutuhkan adalah data numerik
# "cls_id cx cy w h"
# perlu konversi terlebih dahulu
all_annot = []
for item in data:
    if not item:                          # mencegah pemrosesan string yang kosong
        continue                          # lanjut ke iterasi berikutnya

    curr_annot           = item.split()   # melakukan splitting dengan whitespace (" ") sebagai delimiter
    cls_id, cx, cy, w, h = curr_annot     # melakukan unpacking dari list berisikan 5 value ke 5 variabel
    # ['1', '0.5', '0.5', '0.75', '0.75']
    # cls_id, cx, cy, w, h

    # 1 0.5 0.5 0.75 0.75
    # 2 0.3 0.3 0.6 0.6

    # konversi ke data numerik
    cls_id = int(cls_id)
    cx     = float(cx)
    cy     = float(cy)
    w      = float(w)
    h      = float(h)

    all_annot.append((cls_id, cx, cy, w, h))
```
---


##### Memanggil YOLO

Pastikan sudah memasang package `ultralytics` lewat `pip`.

```bash
pip install -U ultralytics
```

Di dalam kode Python, tinggal impor class `YOLO` di dalam package `ultralytics`.

```python
from ultralytics import YOLO

model = YOLO("path/to/model_pt_or_yaml")
```
---


##### Sintaks Penting dalam Class YOLO

Training:
```python
model = YOLO("path/to/model_pt_or_yaml")
model.train(
    data="path/to/data.yaml", # path to your dataset YAML

    imgsz=640,                # ukuran gambar untuk training (bisa disesuaikan)
    
    batch=-1,                 # ukuran batch untuk training paralel (bisa disesuaikan, -1 
                              # berarti YOLO memilih batch sizenya)
    
    epochs=100,               # banyak iterasi terhadap seluruh dataset (N epochs berarti model 
                              # mengulangi dataset sebanyak N kali)
    
    device="cuda:0",          # accelerator yang dipakai (bisa CPU, GPU, bahkan banyak GPU sekaligus)
    
    workers=0,                # banyak thread untuk mengatur data loading dan 
                              # preprocessing (membuat CPU memberikan data lebih cepat      
                              # daripada kecepatan GPU memproses batch sekarang jika bernilai besar, 
                              # tetapi membuat CPU kerja lebih keras)
    
    optimizer="auto",         # tipe optimizer yang dipakai untuk update weights (penting untuk backpropagation)
    
    lr0=0.0001,               # nilai learning rate awal (memengaruhi besar/kecilnya 
                              # weight update setiap kali backward pass, dipengaruhi
                              # oleh pilihan optimizer)
    
    patience=5                # early stopping atau berapa lama model menunggu ketika loss naik atau tidak ada 
                              # perubahan (plateau)
)
```
----

Testing:
```python
model = YOLO("path/to/model_pt_or_yaml")
model.val(
    data="path/to/data.yaml",
    imgsz=640,
    batch=-1,
    device="cuda:0,
    workers=0,
    split="val",               # tipe split yang akan diuji (opsi: "train", "test", dan "val")  
    
    conf=0.5,                  # threshold untuk memfilter hasil prediksi berdasarkan nilai confidence
    
    max_det=300,               # jumlah deteksi maksimum per gambar
    
    half=True                  # flag untuk menggunakan FP16 daripada FP32 (mempercepat 
                               # waktu prediksi, tetapi menjadi kurang akurat)
)
```
---

Inference:
```python
# asumsikan gambar input sudah ada dengan nama variabel `input_image`

model   = YOLO("path/to/model_pt_or_yaml")
outputs = model(input_image, conf=0.5)
```

Setiap kali melakukan inferensi, model akan mengeluarkan sebuah kumpulan prediksi dalam sebuah iterable. Mendapatkan informasi dari setiap prediksi dapat dilakukan dengan cara berikut:

```python
# ambil variabel `outputs` dari contoh di atas

for output in outputs:                              # iterasi per gambar dalam satu batch
    all_detection = output.boxes                    # mendapatkan instance dari class Boxes yang berisikan 
                                                    # seluruh hasil deteksi pada satu gambar

    for detection in all_detection:                 # iterasi tiap deteksi dalam satu gambar 

        coords     = detection.xyxy[0].tolist()     # mengambil koordinat bounding box [x1, y1, x2, y2]

        conf_score = float(detection.conf[0])       # mengambil confidence score

        cls_id     = int(detection.cls[0])          # mengambil ID class

        cls_name   = model.names[cls_id]            # mengubah dari ID class ke nama class
```


#### Contoh Penggunaan YOLO + OpenCV untuk Deteksi Real-Time (Anotasi Manual)


In [ ]:
import cv2                                          # package utama
import torch
from ultralytics import YOLO                        # package utama
from torchvision.transforms import functional as F  # untuk transformasi array NumPy -> tensor

device = "cuda"
if not torch.cuda.is_available():
    device = "cpu"

# load model default
model = YOLO("yolo26n.pt").to(device)

# menginisialisasi kamera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise Exception("Couldn't open the camera.")

while True:
    ret, frame = cap.read()                                 # membaca frame
    if not ret:
        print("Failed to capture the next frame.")
        break
    
    h, w         = frame.shape[:2]

    frame_rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)                        # OpenCV secara native pakai BGR, sehingga perlu dikonversi ke RGB
    frame_tensor = F.to_tensor(frame_rgb).to(device).unsqueeze(0)                # konversi ke tensor
    # R G B -> Batch R G B
    
    results = model(frame_tensor, conf=0.5)                 # prediksi dengan confidence threshold = 0.5
    for result in results:
        boxes = result.boxes
        
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()           # masih ternormalisasi [0, 1]
            conf_score     = box.conf.item()
            cls_id         = box.cls.item()
            cls_name       = model.names[cls_id]
            
            cv2.rectangle(
                frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2
            )
            cv2.putText(
                frame, f"{cls_name} ({conf_score:.2f})", 
                (int(x1), int(y1) - 10), 1, 1, (255, 255, 255)
            )
            
    cv2.imshow("Detection", frame)
    key = cv2.waitKey(1)
    if key == 27:  # ESC key
        break
    
cap.release()
cv2.destroyAllWindows()

# Gambar
# |-> Prediksi
#       |-> Box Prediksi
#               |-> Koordinat + ID Class + Confidence score

#### Contoh Penggunaan YOLO + OpenCV untuk Deteksi Real-Time (Anotasi Dari YOLO)


In [ ]:
import cv2                                          # package utama
import torch
from ultralytics import YOLO                        # package utama
from torchvision.transforms import functional as F  # untuk transformasi array NumPy -> tensor

device = "cuda"
if not torch.cuda.is_available():
    device = "cpu"

# load model default
model = YOLO("yolo26n.pt").to(device)

# menginisialisasi kamera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise Exception("Couldn't open the camera.")

while True:
    ret, frame = cap.read()                                 # membaca frame
    if not ret:
        print("Failed to capture the next frame.")
        break
    
    h, w         = frame.shape[:2]

    frame_rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)                        # OpenCV secara native pakai BGR, sehingga perlu dikonversi ke RGB
    frame_tensor = F.to_tensor(frame_rgb).to(device).unsqueeze(0)                # konversi ke tensor
    
    results = model(frame_tensor, conf=0.5)                 # prediksi dengan confidence threshold = 0.5
    annotated = results[0].plot()
            
    cv2.imshow("Detection", annotated)
    key = cv2.waitKey(1)
    if key == 27:  # ESC key
        break
    
cap.release()
cv2.destroyAllWindows()


### ROS Integration


ROS Setup
```bash
mkdir ROS/src -p
cd src
catkin_create_pkg opencv rospy sensor_msgs cv_bridge image_transport
cd ..
catkin_make
```


In [ ]:
# scripts/publisher.py
import rospy
from sensor_msgs.msg import Image
from cv_bridge import CvBridge
import cv2

def main():
    rospy.init_node('camera_publisher', anonymous=True)
    pub = rospy.Publisher('/camera/image_raw', Image, queue_size=10)
    bridge = CvBridge()
    rate = rospy.Rate(30)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        rospy.logerr("Camera not detected.")
        return

    while not rospy.is_shutdown():
        ret, frame = cap.read()
        if not ret:
            rospy.logwarn("Failed to grab frame.")
            continue

        msg = bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        pub.publish(msg)
        rospy.loginfo("Published frame.")
        rate.sleep()

    cap.release()

if __name__ == '__main__':
    try:
        main()
    except rospy.ROSInterruptException:
        pass


In [ ]:
# scripts/subscriber.py
import rospy
from sensor_msgs.msg import Image
from cv_bridge import CvBridge, CvBridgeError
import cv2

bridge = CvBridge()

def callback(msg):
    try:
        frame = bridge.imgmsg_to_cv2(msg, "bgr8")
    except CvBridgeError as e:
        rospy.logerr(f"CV Bridge error: {e}")
        return

    cv2.imshow("Camera Feed", frame)
    key = cv2.waitKey(1)
    if key == 27:  # ESC key
        rospy.signal_shutdown("ESC pressed.")

def main():
    rospy.init_node('image_subscriber', anonymous=True)
    rospy.Subscriber('/camera/image_raw', Image, callback)
    rospy.loginfo("Image subscriber started.")
    rospy.spin()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()


jangan lupa untuk membenahi bagian CMakeList.txt untuk python depedenciesnya seperti
```cmake
catkin_install_python(PROGRAMS
 scripts/publisher.py
 scripts/subscriber.py
 DESTINATION ${CATKIN_PACKAGE_BIN_DESTINATION}
)
```
